下载模型
```shell
hf download lmms-lab/LLaVA-OneVision-1.5-8B-Instruct --local-dir ~/Study/models/llava/LLaVA-OneVision-1.5-8B-Instruct
```

## 模型使用

In [ ]:
import torch
import os
from transformers import AutoProcessor, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)

models_dir = "/mnt/c/Users/Alvis/Study/models"
model_name = "LLaVA-OneVision-1.5-8B-Instruct"

model_path = f"{models_dir}/{model_name}"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype="auto", device_map="auto", trust_remote_code=True,
)

The tokenizer you are loading from '/mnt/c/Users/Alvis/Study/models/LLaVA-OneVision-1.5-8B-Instruct' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [01:46<00:00, 26.71s/it]
/home/alvis/miniconda3/envs/llava/lib/python3.10/site-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['model.visual.class_embedding', 'model.visual.class_pos_emb']
  warnings.warn(
Some parameters are on the meta device because they were offloaded to the cpu.


In [7]:
messages = [
    {
        "role": "user",
        "content": [
            # https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG
            {"type": "image", "url": "./assets/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=40)
generated_text = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)
print(generated_text)

The candy features an illustration of a turtle, which can be identified by its distinctive shell and limbs.


## 模型架构

In [12]:
language_model = model.language_model
print(language_model)

LLaVAOneVision1_5_TextModel(
  (embed_tokens): Embedding(151936, 4096)
  (layers): ModuleList(
    (0-35): 36 x LLaVAOneVision1_5_DecoderLayer(
      (self_attn): LLaVAOneVision1_5_SdpaAttention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (q_norm): LLaVAOneVision1_5_RMSNorm((128,), eps=1e-06)
        (k_norm): LLaVAOneVision1_5_RMSNorm((128,), eps=1e-06)
      )
      (mlp): LLaVAOneVision1_5_MLP(
        (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): LLaVAOneVision1_5_RMSNorm((4096,), eps

In [15]:
visual = model.visual
print(visual)

RiceTransformerPretrainedModel(
  (patch_embed): RicePatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
  )
  (rotary_pos_emb): RiceRotaryEmbedding()
  (pre_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (blocks): ModuleList(
    (0-23): 24 x RiceBlock(
      (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (attn): RiceSdpaAttention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
      )
      (mlp): RiceMlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELUActivation()
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
      )
    )
  )
  (merger): RicePatchMerger(
    (ln_q): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (mlp): Sequential(
      (0): Linear(in_features=4